# Practice: curvilinear and rectilinear RTN

A sandbox for the residual conversion in `orbit_correction.residuals`. It takes a real
element set (the ISS, from the committed CelesTrak-derived fixture), gets its SGP4 state,
places a "truth" position at known offsets from it, and compares the two ways of
expressing the difference:

1. the frame built from the cheap (SGP4) state;
2. rectilinear against curvilinear RTN as the along-track separation grows;
3. the round trip (forward, then inverse);
4. what the conversion refuses, and why (angle wrapping);
5. a week of a cheap-against-"truth" history, with SGP4 at a different B* standing in
   for truth (no JVM needed).

Nothing here writes to `src/` or to project data. Edit the parameters (separations, B*
factor) or the local copies of the functions and rerun.

In [1]:
from pathlib import Path

import numpy as np
import pyarrow as pa

from orbit_correction import catalog
from orbit_correction import cheap_prop as C
from orbit_correction import residuals as R

REPO = Path.cwd().parent if Path.cwd().name == "practice" else Path.cwd()
elsets = catalog.load_celestrak_omm(REPO / "tests/fixtures/celestrak_gp_sample.json")
iss = elsets.filter(pa.compute.equal(elsets.column("norad_id"), 25544))
start = C.sgp4_epoch_states(iss)[0].trajectory
r_c, v_c = start.r_gcrf_km[0], start.v_gcrf_km_s[0]
rc = np.linalg.norm(r_c)
print(f"ISS SGP4 state at epoch: |r| = {rc:.3f} km, |v| = {np.linalg.norm(v_c):.4f} km/s")

ISS SGP4 state at epoch: |r| = 6794.004 km, |v| = 7.6645 km/s


## 1. The frame comes from the cheap state

`u_r` points along the cheap position, `n` along the cheap orbit's angular momentum, and
`u_t = n x u_r` completes the triad in the direction of motion. It is built from the
cheap state alone because, when a trained model is used in screening, the cheap state is
all there is: the frame has to be reconstructible without truth.

In [2]:
rcn, u_r, u_t, n = R.cheap_frame(r_c[None], v_c[None])
axes = np.vstack([u_r, u_t, n])
print("axes (rows u_r, u_t, n):\n", np.round(axes, 6))
print("orthonormal:", np.allclose(axes @ axes.T, np.eye(3), atol=1e-14))
# u_t is not exactly along v for an eccentric orbit: v has a small radial part
gamma = np.degrees(np.arcsin(np.dot(v_c, u_r[0]) / np.linalg.norm(v_c)))
print(f"flight-path angle (v against u_t): {gamma:.4f} deg")

axes (rows u_r, u_t, n):
 [[-0.581886 -0.813269  0.001552]
 [ 0.506641 -0.361003  0.782938]
 [-0.636179  0.456367  0.622098]]
orthonormal: True
flight-path angle (v against u_t): -0.0841 deg


## 2. Rectilinear against curvilinear as the along-track separation grows

Put truth on the cheap orbit's circle (same radius, in the plane) an arc `s` behind the
cheap position. The physical error is purely along-track. Curvilinear RTN says exactly
that: R = 0, T = s, N = 0. Rectilinear RTN projects the straight chord `r_c - r_t` on the
axes; the chord cuts inside the circle, so part of it lands on the radial axis:
`R_rect = r (1 - cos(s/r)) ~ s^2 / (2 r)`, and T comes out short (`r sin(s/r)`).

Change `separations_km` to explore.

In [3]:
separations_km = [1, 10, 100, 500, 1000, 2000, 5000]
print(f"{'s km':>6} | {'curv R':>8} {'curv T':>9} {'curv N':>7} | "
      f"{'rect R':>9} {'rect T':>9} | {'s^2/2r':>9}")
for s in separations_km:
    r_t = R.states_from_rtn(r_c, v_c, np.array([0.0, s, 0.0]))
    cur = R.rtn_from_states(r_c, v_c, r_t)
    rec = R.rectilinear_rtn(r_c, v_c, r_t)
    print(f"{s:6.0f} | {cur[0]:8.2e} {cur[1]:9.3f} {cur[2]:7.0e} | "
          f"{rec[0]:9.4f} {rec[1]:9.3f} | {s**2 / (2 * rc):9.4f}")

  s km |   curv R    curv T  curv N |    rect R    rect T |    s^2/2r
     1 | 9.09e-13     1.000  -0e+00 |    0.0001     1.000 |    0.0001
    10 | 0.00e+00    10.000  -0e+00 |    0.0074    10.000 |    0.0074
   100 | 0.00e+00   100.000  -5e-13 |    0.7359    99.996 |    0.7359
   500 | 9.09e-13   500.000  -5e-13 |   18.3903   499.549 |   18.3986
  1000 | 9.09e-13  1000.000  -5e-13 |   73.4615   996.393 |   73.5943
  2000 | 9.09e-13  2000.000   9e-13 |  292.2575  1971.239 |  294.3772
  5000 | -9.09e-13  5000.000  -0e+00 | 1758.3016  4560.723 | 1839.8576


At 1,000 km of along-track error (a week of SGP4 error for a ~350 km object is about that)
rectilinear RTN reports about 73 km of radial error at the ISS's radius that does not exist. A model trained on
rectilinear targets would have to learn that artefact; one trained on curvilinear targets
learns the along-track drift as along-track drift.

At small separations the two agree to second order. Mix in radial and cross-track parts and
watch the difference scale as `s^2 / r`:

In [4]:
rng = np.random.default_rng(0)
d = rng.normal(size=3)
d /= np.linalg.norm(d)
for s in [0.01, 0.1, 1, 10, 100]:
    r_t = r_c - s * d
    diff = np.abs(R.rtn_from_states(r_c, v_c, r_t) - R.rectilinear_rtn(r_c, v_c, r_t)).max()
    print(f"s = {s:7.2f} km: max |curvilinear - rectilinear| = {diff:.3e} km "
          f"= {diff / (s**2 / rc):.3f} x s^2/r")

s =    0.01 km: max |curvilinear - rectilinear| = 7.339e-09 km = 0.499 x s^2/r
s =    0.10 km: max |curvilinear - rectilinear| = 7.339e-07 km = 0.499 x s^2/r
s =    1.00 km: max |curvilinear - rectilinear| = 7.339e-05 km = 0.499 x s^2/r
s =   10.00 km: max |curvilinear - rectilinear| = 7.339e-03 km = 0.499 x s^2/r
s =  100.00 km: max |curvilinear - rectilinear| = 7.344e-01 km = 0.499 x s^2/r


## 3. The round trip

The inverse turns (R, T, N) back into a position using only the cheap state:
`rho = |r_c| - R`, `du = T / |r_c|`, `phi = -N / |r_c|`, then rotate `u_r` by `-du` about `n`
and tilt by `phi` towards `n`. It is exact (a spherical coordinate system attached to the
cheap state), so forward then inverse returns the input to rounding error.

In [5]:
n_states = 10_000
rtn = np.column_stack([rng.uniform(-50, 50, n_states), rng.uniform(-2000, 2000, n_states),
                       rng.uniform(-50, 50, n_states)])
rr = np.tile(r_c, (n_states, 1))
vv = np.tile(v_c, (n_states, 1))
r_t = R.states_from_rtn(rr, vv, rtn)
back = R.rtn_from_states(rr, vv, r_t)
print(f"max |RTN -> position -> RTN| error: {np.abs(back - rtn).max():.2e} km")

max |RTN -> position -> RTN| error: 3.16e-12 km


## 4. Angle wrapping: why the conversion refuses |du| > pi/2

The along-track angle comes from `atan2`, which returns values in (-pi, pi]. A truth position
more than half an orbit behind looks, from one instant, exactly like one less than half an
orbit ahead. Near pi the target flips sign. Below is a local copy of the forward
conversion **without** the limit (edit it freely; it doesn't touch `src/`).

In [6]:
def t_component_unlimited(r_c, v_c, r_t):
    rcn, u_r, u_t, n = R.cheap_frame(r_c[None], v_c[None])
    x, y = r_t @ u_r[0], r_t @ u_t[0]
    return rcn[0] * np.arctan2(-y, x)

for frac in [0.40, 0.49, 0.499, 0.501, 0.51, 0.60]:
    arc = frac * 2 * np.pi * rc               # true arc behind, km
    ang = arc / rc
    r_t = rc * (np.cos(ang) * u_r[0] - np.sin(ang) * u_t[0])
    t_km = t_component_unlimited(r_c, v_c, r_t)
    print(f"truth {frac:5.3f} orbit behind (arc {arc:8.0f} km): T = {t_km:9.0f} km")
try:
    R.rtn_from_states(r_c, v_c, -r_c)
except R.RTNDomainError as e:
    print("\nresiduals.rtn_from_states refuses it:", e)

truth 0.400 orbit behind (arc    17075 km): T =     17075 km
truth 0.490 orbit behind (arc    20917 km): T =     20917 km
truth 0.499 orbit behind (arc    21301 km): T =     21301 km
truth 0.501 orbit behind (arc    21387 km): T =    -21301 km
truth 0.510 orbit behind (arc    21771 km): T =    -20917 km
truth 0.600 orbit behind (arc    25613 km): T =    -17075 km

residuals.rtn_from_states refuses it: |du| up to 3.142 rad in 1 row(s), above 1.571: along-track separation too large for an unambiguous angle


## 5. A week of residuals, with SGP4 at a different B* standing in for truth

Real truth needs Orekit (a JVM). To see the shape of a residual history here, use SGP4 itself
with B* multiplied by `bstar_factor` as a stand-in for "truth": a drag mismatch, which is
the largest part of the real residual at low altitude. Both run on the proposed horizon grid
(`config/residuals.toml`).

In [7]:
bstar_factor = 1.5
cfg = R.load_residual_config()
grid = R.horizon_grid(cfg)
rec = iss.to_pylist()[0]
rec["bstar_per_earth_radii"] *= bstar_factor
stand_in = pa.Table.from_pylist([rec], schema=iss.schema)
cheap = C.propagate_sgp4_from_epoch(iss, grid)[0].trajectory
other = C.propagate_sgp4_from_epoch(stand_in, grid)[0].trajectory
cur = R.rtn_from_states(cheap.r_gcrf_km, cheap.v_gcrf_km_s, other.r_gcrf_km)
rec_ = R.rectilinear_rtn(cheap.r_gcrf_km, cheap.v_gcrf_km_s, other.r_gcrf_km)
print(f"{grid.size} horizons per window")
print(f"{'dt':>9} | {'curv R':>8} {'curv T':>9} {'curv N':>7} | {'rect R':>8}")
for h in [60, 3600, 86400, 3 * 86400, 7 * 86400]:
    j = int(np.searchsorted(grid, h))
    print(f"{grid[j] / 86400:7.3f} d | {cur[j, 0]:8.3f} {cur[j, 1]:9.3f} {cur[j, 2]:7.3f} | "
          f"{rec_[j, 0]:8.3f}")

199 horizons per window
       dt |   curv R    curv T  curv N |   rect R
  0.001 d |    0.000    -0.000   0.000 |    0.000
  0.042 d |    0.001    -0.002   0.000 |    0.001
  1.050 d |    0.020    -1.758   0.000 |    0.021
  3.010 d |    0.066   -14.421   0.006 |    0.081
  7.000 d |    0.049   -78.239   0.069 |    0.500


T is negative: the stand-in has more drag, so it sinks and speeds up and runs *ahead* of the
cheap position (cheap behind gives T < 0). By 7 days rectilinear RTN already puts several
hundred metres of the along-track error into R; the curvilinear R stays at tens of metres,
the actual difference in radius.

Things to try:

- raise `bstar_factor` (or use the low Starlink-1008, NORAD 44714) until the 7-day
  along-track error passes a few hundred km, and watch the rectilinear radial column grow
  while the curvilinear one doesn't;
- change the grid in a copy of `cfg` (`dataclasses.replace(cfg, log_points=20)`) and see
  which horizons survive;
- build the frame from the *truth* state instead of the cheap one in a copy of section 4's
  function, and note that the inverse would then need truth, which screening doesn't have.